# CPP Low-Rank RNN Rank-5 Latent-Dynamics Pipeline — Self-Contained Notebook

This notebook trains a Rank=5 `CPPLowRankRNN` on response-locked CPP EEG trials, then runs held-out testing, latent export, diagnostics, and ridge regression. It is self-contained: all classes and functions needed for the pipeline are defined in the cells below.

| # | Stage | Key output inside this run folder |
|---|-------|-----------|
| 0 | Imports & path setup | `tmp/low_rank_r5_notebook_runs/<timestamp>/` |
| 1 | Configuration classes | — |
| 2 | Utility functions | — |
| 3 | Data contract validator | `Results/low_rank_r5_validation/` |
| 4 | Dataset pipeline | — |
| 5 | Model: CPPLowRankRNN Rank=5 + loss | — |
| 6 | Training and testing pipeline | — |
| ▶ 7 | Run low-rank Rank=5 training | `Results/low_rank_r5_model_checkpoints/best_low_rank_r5_model.pt` |
| ▶ 8 | Test best Rank=5 model | `Results/low_rank_r5_model_checkpoints/test_metrics.json` |
| ▶ 9 | Export Rank=5 latent states | `Data/IntermediateData/latents_low_rank_r5/latents_low_rank_r5.npz` |
| 10 | Low-rank Rank=5 latent diagnostics | `Results/low_rank_r5_diagnostics/` |
| ▶ 11 | Ridge regression with Rank=5 latents | `Results/low_rank_r5_regression/` |
| 12 | Results & next steps | printed summary |

The paths above are relative to the temporary run directory created by this notebook, so existing project outputs are not overwritten.

## 0 · Imports & Path Setup

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
import json
import random
import shutil
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import Image, Markdown, display
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "Data" / "ProcessedData").exists() and (candidate / "Scripts").exists():
        PROJECT_ROOT = candidate
        break

DATASET_DIR = PROJECT_ROOT / "Data" / "ProcessedData"
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "tmp" / "low_rank_r5_notebook_runs" / RUN_STAMP

RUN_DATA_ROOT = RUN_ROOT / "Data"
RUN_RESULTS_ROOT = RUN_ROOT / "Results"
LOW_RANK_R5_CHECKPOINT_DIR = RUN_RESULTS_ROOT / "low_rank_r5_model_checkpoints"
LOW_RANK_R5_BEST_CKPT = LOW_RANK_R5_CHECKPOINT_DIR / "best_low_rank_r5_model.pt"
LOW_RANK_R5_TEST_METRICS = LOW_RANK_R5_CHECKPOINT_DIR / "test_metrics.json"
LOW_RANK_R5_VALIDATION_DIR = RUN_RESULTS_ROOT / "low_rank_r5_validation"
LOW_RANK_R5_DIAGNOSTICS_DIR = RUN_RESULTS_ROOT / "low_rank_r5_diagnostics"
LOW_RANK_R5_REGRESSION_DIR = RUN_RESULTS_ROOT / "low_rank_r5_regression"
LOW_RANK_R5_LATENT_DIR = RUN_DATA_ROOT / "IntermediateData" / "latents_low_rank_r5"
LOW_RANK_R5_LATENT_PATH = LOW_RANK_R5_LATENT_DIR / "latents_low_rank_r5.npz"

for path in [
    LOW_RANK_R5_CHECKPOINT_DIR,
    LOW_RANK_R5_VALIDATION_DIR,
    LOW_RANK_R5_DIAGNOSTICS_DIR,
    LOW_RANK_R5_REGRESSION_DIR,
    LOW_RANK_R5_LATENT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}  (exists={DATASET_DIR.exists()})")
print(f"Run root     : {RUN_ROOT}")
print(f"Checkpoint   : {LOW_RANK_R5_BEST_CKPT}")
print(f"Latents      : {LOW_RANK_R5_LATENT_PATH}")

## 1 · Configuration Classes

In [ ]:
@dataclass(frozen=True)
class DataContractConfig:
    expected_files: Tuple[str, ...] = (
        "eeg_cpp_trials.npy",
        "metadata.csv",
        "times_ms.npy",
        "channel_names.txt",
        "preprocessing_notes.md",
    )
    expected_channel_order: Tuple[str, ...] = ("CP1", "CP2", "CPz")
    required_metadata_columns: Tuple[str, ...] = ("trial_id", "alignment")
    optional_aliases: dict = field(default_factory=dict)


@dataclass(frozen=True)
class LowRankRNNConfig:
    rank: int = 5
    population_dim: int = 64
    input_scale: float = 1.0
    recurrent_scale: float = 1.0
    state_leak: float = 0.25


@dataclass(frozen=True)
class LossWeights:
    lambda_recon: float = 1.0
    lambda_future: float = 0.2
    lambda_derivative: float = 0.5
    lambda_variance: float = 0.5
    lambda_cpp_mean: float = 0.5
    lambda_cpp_prior: float = 0.1
    lambda_monotonic: float = 1.0
    lambda_slope_floor: float = 0.5
    lambda_late_amplitude: float = 1.0
    lambda_cpp_mean_alignment: float = 0.05
    lambda_smooth: float = 0.001
    future_weight_scale: float = 0.75
    slope_floor_ratio: float = 0.5
    enable_cpp_shape_prior: bool = True
    analysis_window_ms: Tuple[float, float] = (-600.0, -50.0)
    late_window_ms: Tuple[float, float] = (-120.0, -50.0)


@dataclass(frozen=True)
class TrainingConfig:
    seed: int = 42
    batch_size: int = 64
    train_fraction: float = 0.70
    val_fraction: float = 0.15
    test_fraction: float = 0.15
    future_horizon_ms: int = 50
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0
    max_epochs: int = 100
    early_stopping_patience: int = 15
    analysis_window_ms: Tuple[float, float] = (-600.0, -50.0)
    early_window_ms: Tuple[float, float] = (-600.0, -300.0)
    mid_window_ms: Tuple[float, float] = (-300.0, -120.0)
    late_window_ms: Tuple[float, float] = (-120.0, -50.0)
    model: LowRankRNNConfig = field(default_factory=LowRankRNNConfig)
    loss: LossWeights = field(default_factory=LossWeights)


@dataclass(frozen=True)
class AnalysisConfig:
    response_locked_window_ms: Tuple[int, int] = (-600, -50)
    contaminated_window_ms: Tuple[int, int] = (-50, 100)
    pca_components: int = 3
    rt_bin_quantiles: Tuple[float, float] = (0.33, 0.66)
    evidence_bin_quantiles: Tuple[float, ...] = (0.50,)

## 2 · Utility Functions

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if hasattr(value, "__dataclass_fields__"):
        return asdict(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serialisable")


def write_json(path: Path, payload: Dict[str, Any]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True, default=_json_default)


def safe_float(value: Any) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def aggregate_metric_dicts(rows: List[Dict[str, float]]) -> Dict[str, float]:
    if not rows:
        return {}
    keys = sorted({key for row in rows for key in row})
    return {key: float(np.mean([row[key] for row in rows if key in row])) for key in keys}


def display_saved_figure(path: Path, title: str) -> None:
    if path.exists():
        print(title)
        display(Image(filename=str(path)))
    else:
        print(f"Missing figure: {path}")


def save_dataframe(path: Path, df: pd.DataFrame) -> None:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)


NATURE_BLUE = "#3B5BA7"
NATURE_TEAL = "#007A87"
NATURE_ORANGE = "#D55E00"
NATURE_RED = "#B2182B"
NATURE_GRAY = "#8A8A8A"
NATURE_DARK = "#222222"
NATURE_LIGHT_GRAY = "#D9D9D9"
NATURE_PALETTE = [NATURE_BLUE, NATURE_TEAL, NATURE_ORANGE, NATURE_RED, NATURE_GRAY]


def set_publication_style() -> None:
    import matplotlib as mpl

    mpl.rcParams.update(
        {
            "figure.facecolor": "white",
            "axes.facecolor": "white",
            "savefig.facecolor": "white",
            "savefig.edgecolor": "white",
            "axes.edgecolor": NATURE_DARK,
            "axes.linewidth": 0.8,
            "axes.grid": False,
            "font.size": 8,
            "axes.labelsize": 9,
            "axes.titlesize": 9,
            "xtick.labelsize": 8,
            "ytick.labelsize": 8,
            "legend.fontsize": 8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "axes.prop_cycle": mpl.cycler(color=NATURE_PALETTE),
        }
    )


def clean_axes(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(width=0.8, length=3, color=NATURE_DARK)
    ax.grid(False)


def save_publication_figure(fig: plt.Figure, output_base: Path, dpi: int = 600) -> None:
    ensure_dir(output_base.parent)
    for ext in ("pdf", "svg", "png"):
        kwargs = {"bbox_inches": "tight", "facecolor": "white"}
        if ext == "png":
            kwargs["dpi"] = dpi
        fig.savefig(output_base.with_suffix(f".{ext}"), **kwargs)


def bootstrap_ci(values: Sequence[float], n_boot: int = 5000, seed: int = 42) -> Tuple[float, float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return float("nan"), float("nan"), float("nan")
    if len(arr) == 1:
        return float(arr[0]), float(arr[0]), float(arr[0])
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)])
    return float(arr.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


set_publication_style()

## 3 · Data Contract Validator

In [ ]:
def _resolve_required_columns(metadata: pd.DataFrame, config: DataContractConfig) -> Tuple[pd.DataFrame, List[str]]:
    renamed = metadata.copy()
    missing: List[str] = []
    for required in config.required_metadata_columns:
        if required in renamed.columns:
            continue
        aliases = config.optional_aliases.get(required, ())
        alias_match = next((alias for alias in aliases if alias in renamed.columns), None)
        if alias_match is None:
            if required == "alignment":
                renamed[required] = "response_locked"
                continue
            missing.append(required)
            continue
        renamed = renamed.rename(columns={alias_match: required})
    return renamed, missing


def _read_channel_names(path: Path) -> List[str]:
    with path.open("r", encoding="utf-8") as handle:
        return [line.strip() for line in handle.readlines() if line.strip()]


def _extract_sampling_rate(times_ms: np.ndarray) -> float:
    if len(times_ms) < 2:
        return float("nan")
    step_ms = float(np.mean(np.diff(times_ms)))
    return float("nan") if step_ms == 0 else 1000.0 / step_ms


def validate_stage2_dataset(
    dataset_dir: Path,
    output_dir: Path,
    config: DataContractConfig | None = None,
) -> Dict[str, object]:
    config = config or DataContractConfig()
    output_dir = ensure_dir(output_dir)
    file_checks = {name: (dataset_dir / name).exists() for name in config.expected_files}
    missing_files = [name for name, exists in file_checks.items() if not exists]
    report: Dict[str, object] = {
        "dataset_dir": str(dataset_dir),
        "contract": asdict(config),
        "file_checks": file_checks,
        "missing_files": missing_files,
        "passed": False,
    }
    if missing_files:
        write_json(output_dir / "stage1_blocking_issue_report.json", report)
        return report

    eeg = np.load(dataset_dir / "eeg_cpp_trials.npy")
    times_ms = np.load(dataset_dir / "times_ms.npy")
    metadata = pd.read_csv(dataset_dir / "metadata.csv")
    metadata, missing_columns = _resolve_required_columns(metadata, config)
    channels = _read_channel_names(dataset_dir / "channel_names.txt")
    notes_text = (dataset_dir / "preprocessing_notes.md").read_text(encoding="utf-8")

    n_trials, n_timepoints, n_channels = eeg.shape
    report.update(
        {
            "shape_summary": {
                "n_trials": int(n_trials),
                "n_timepoints": int(n_timepoints),
                "n_channels": int(n_channels),
            },
            "metadata_rows_match": len(metadata) == n_trials,
            "times_match": len(times_ms) == n_timepoints,
            "channel_order_matches": tuple(channels) == config.expected_channel_order,
            "missing_metadata_columns": missing_columns,
            "sampling_rate_hz": _extract_sampling_rate(times_ms),
            "preprocessing_policy_extract": {
                "reference_mentioned": "reference" in notes_text.lower(),
                "filter_mentioned": "filter" in notes_text.lower(),
                "artifact_mentioned": "artifact" in notes_text.lower() or "ica" in notes_text.lower(),
                "baseline_mentioned": "baseline" in notes_text.lower(),
            },
        }
    )
    report["passed"] = all(
        [
            report["metadata_rows_match"],
            report["times_match"],
            report["channel_order_matches"],
            not missing_columns,
        ]
    )
    write_json(output_dir / "stage1_data_contract_report.json", report)
    return report


_validation_report = validate_stage2_dataset(DATASET_DIR, LOW_RANK_R5_VALIDATION_DIR)
print(json.dumps(_validation_report["shape_summary"], indent=2))
if not _validation_report["passed"]:
    raise RuntimeError("Dataset contract validation failed. See the validation report.")

## 4 · Dataset Pipeline

The dataset pipeline loads response-locked EEG, z-normalises channels with training-split statistics, builds one-step future targets, creates a valid time mask, and returns train/validation/test dataloaders.

In [ ]:
@dataclass
class Stage2SplitArtifacts:
    train_indices: np.ndarray
    val_indices: np.ndarray
    test_indices: np.ndarray
    train_mean: np.ndarray
    train_std: np.ndarray
    horizon_steps: int


def _coerce_low_rank_config(value: Any) -> LowRankRNNConfig:
    if isinstance(value, LowRankRNNConfig):
        return value
    if isinstance(value, dict):
        return LowRankRNNConfig(**value)
    raise TypeError(f"Unsupported low-rank config type: {type(value)!r}")


def _coerce_loss_weights(value: Any) -> LossWeights:
    if isinstance(value, LossWeights):
        return value
    if isinstance(value, dict):
        return LossWeights(**value)
    raise TypeError(f"Unsupported loss config type: {type(value)!r}")


def _coerce_training_config(config: Any) -> TrainingConfig:
    if isinstance(config, TrainingConfig):
        return config
    if isinstance(config, dict):
        raw = dict(config)
    elif hasattr(config, "__dict__"):
        raw = dict(vars(config))
    else:
        raise TypeError(f"Unsupported training config type: {type(config)!r}")
    training_fields = set(TrainingConfig.__dataclass_fields__)
    model_source = raw.pop("model", LowRankRNNConfig())
    loss_source = raw.pop("loss", LossWeights())
    clean = {key: value for key, value in raw.items() if key in training_fields and key not in {"model", "loss"}}
    clean["model"] = _coerce_low_rank_config(model_source)
    clean["loss"] = _coerce_loss_weights(loss_source)
    return TrainingConfig(**clean)


def _random_trial_split(n_trials: int, config: TrainingConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    total = config.train_fraction + config.val_fraction + config.test_fraction
    if not np.isclose(total, 1.0):
        raise ValueError(f"Split fractions must sum to 1.0, got {total}")
    rng = np.random.default_rng(config.seed)
    indices = np.arange(n_trials)
    rng.shuffle(indices)
    n_train = int(round(n_trials * config.train_fraction))
    n_val = int(round(n_trials * config.val_fraction))
    train_idx = indices[:n_train]
    val_idx = indices[n_train : n_train + n_val]
    test_idx = indices[n_train + n_val :]
    return train_idx, val_idx, test_idx


def _compute_channel_stats(eeg: np.ndarray, train_idx: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    train = eeg[train_idx]
    mean = train.reshape(-1, train.shape[-1]).mean(axis=0).astype(np.float32)
    std = train.reshape(-1, train.shape[-1]).std(axis=0).astype(np.float32)
    std[std < 1e-6] = 1.0
    return mean, std


def _normalize_with_stats(eeg: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((eeg - mean[None, None, :]) / std[None, None, :]).astype(np.float32)


def _build_future_targets(eeg: np.ndarray, horizon_steps: int) -> np.ndarray:
    targets = np.zeros_like(eeg, dtype=np.float32)
    if eeg.shape[1] > horizon_steps:
        targets[:, :-horizon_steps, :] = eeg[:, horizon_steps:, :]
    return targets


def _build_trial_mask(
    times_ms: np.ndarray,
    n_trials: int,
    config: TrainingConfig,
    horizon_steps: int,
) -> np.ndarray:
    valid_time = (times_ms >= config.analysis_window_ms[0]) & (times_ms <= config.analysis_window_ms[1])
    if horizon_steps > 0:
        valid_time[-horizon_steps:] = False
    return np.repeat(valid_time[None, :], n_trials, axis=0).astype(np.float32)


def _build_time_weights(times_ms: np.ndarray, config: TrainingConfig) -> np.ndarray:
    weights = np.zeros_like(times_ms, dtype=np.float32)
    early = (times_ms >= config.early_window_ms[0]) & (times_ms < config.early_window_ms[1])
    mid = (times_ms >= config.mid_window_ms[0]) & (times_ms < config.mid_window_ms[1])
    late = (times_ms >= config.late_window_ms[0]) & (times_ms <= config.late_window_ms[1])
    weights[early] = 1.0
    weights[mid] = 1.75
    weights[late] = 2.5
    return weights


class EEGWindowDataset(Dataset):
    def __init__(
        self,
        eeg: np.ndarray,
        targets: np.ndarray,
        mask: np.ndarray,
        times_ms: np.ndarray,
        indices: np.ndarray,
    ) -> None:
        self.eeg = torch.as_tensor(eeg[indices], dtype=torch.float32)
        self.targets = torch.as_tensor(targets[indices], dtype=torch.float32)
        self.mask = torch.as_tensor(mask[indices], dtype=torch.float32)
        self.times_ms = torch.as_tensor(times_ms, dtype=torch.float32)
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self) -> int:
        return int(self.eeg.shape[0])

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {
            "eeg": self.eeg[idx],
            "target_future": self.targets[idx],
            "mask": self.mask[idx],
            "times_ms": self.times_ms,
            "trial_idx": self.indices[idx],
        }


def load_stage2_dataset(
    dataset_dir: Path,
    config: TrainingConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    config = _coerce_training_config(config)
    eeg = np.load(dataset_dir / "eeg_cpp_trials.npy").astype(np.float32)
    if not np.isfinite(eeg).all():
        eeg = np.nan_to_num(eeg, nan=0.0, posinf=0.0, neginf=0.0)
    times_ms = np.load(dataset_dir / "times_ms.npy").astype(np.float32)
    metadata = pd.read_csv(dataset_dir / "metadata.csv")
    metadata, missing_columns = _resolve_required_columns(metadata, DataContractConfig())
    if missing_columns:
        raise ValueError(f"Missing required metadata columns: {missing_columns}")
    channels = _read_channel_names(dataset_dir / "channel_names.txt")
    if tuple(channels) != DataContractConfig().expected_channel_order:
        raise ValueError(f"Unexpected channel order: {channels}")

    train_idx, _, _ = _random_trial_split(len(metadata), config)
    train_mean, train_std = _compute_channel_stats(eeg, train_idx)
    eeg_norm = _normalize_with_stats(eeg, train_mean, train_std)

    fs = 1000.0 / float(np.mean(np.diff(times_ms)))
    horizon_steps = max(1, int(round(config.future_horizon_ms * fs / 1000.0)))
    targets = _build_future_targets(eeg_norm, horizon_steps)
    mask = _build_trial_mask(times_ms, eeg_norm.shape[0], config, horizon_steps)
    return eeg_norm, targets, mask, times_ms, metadata


def make_dataloaders(
    eeg: np.ndarray,
    targets: np.ndarray,
    mask: np.ndarray,
    times_ms: np.ndarray,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader, DataLoader, Dict[str, np.ndarray]]:
    config = _coerce_training_config(config)
    set_seed(config.seed)
    train_idx, val_idx, test_idx = _random_trial_split(eeg.shape[0], config)
    weighted_mask = mask * _build_time_weights(times_ms, config)[None, :]

    def make_loader(indices: np.ndarray, shuffle: bool) -> DataLoader:
        ds = EEGWindowDataset(eeg, targets, weighted_mask, times_ms, indices)
        return DataLoader(ds, batch_size=config.batch_size, shuffle=shuffle)

    return (
        make_loader(train_idx, True),
        make_loader(val_idx, False),
        make_loader(test_idx, False),
        {"train": train_idx, "val": val_idx, "test": test_idx},
    )

## 5 · Model Architecture — CPPLowRankRNN Rank=5

The recurrent state is a rank-dimensional low-rank latent state. This notebook fixes `rank = 5`, so each trial and each time point gets five rank-5 latent variables: z1, z2, z3, z4, z5. These z variables are the core objects for later CPP, RT, condition, and evidence-strength analyses.

```
Input EEG: (B, T, C)
    ↓
Linear input_to_population: C → population_dim
    ↓
Add recurrent low-rank drive from previous z
    ↓
tanh nonlinear population
    ↓
n_factor projection: population_dim → rank
    ↓
leaky update
    ↓
latents: (B, T, rank), where rank = 5
    ↓
recon_head: rank → C
    ↓
pred_head: rank → C
```

z1–z5 are not assumed in advance to equal a specific CPP component; their roles are tested downstream through regression, correlation, and diagnostics.

In [ ]:
@dataclass
class ForwardOutputs:
    reconstructed: torch.Tensor
    predicted: torch.Tensor
    latents: torch.Tensor


class CPPLowRankRNN(nn.Module):
    def __init__(self, n_channels: int, config: LowRankRNNConfig) -> None:
        super().__init__()
        if config.rank != 5:
            raise ValueError("This notebook is fixed to rank=5.")
        if config.population_dim < config.rank:
            raise ValueError("population_dim must be >= rank")
        if not 0.0 < config.state_leak <= 1.0:
            raise ValueError("state_leak must be in (0, 1]")
        self.n_channels = n_channels
        self.cfg = config
        self.input_to_population = nn.Linear(n_channels, config.population_dim)
        self.m_factor = nn.Parameter(torch.randn(config.population_dim, config.rank) * 0.15)
        self.n_factor = nn.Parameter(torch.randn(config.population_dim, config.rank) * 0.15)
        self.state_bias = nn.Parameter(torch.zeros(config.rank))
        self.recon_head = nn.Sequential(
            nn.LayerNorm(config.rank),
            nn.Linear(config.rank, config.rank),
            nn.Tanh(),
            nn.Linear(config.rank, n_channels),
        )
        self.pred_head = nn.Sequential(
            nn.LayerNorm(config.rank),
            nn.Linear(config.rank, config.rank),
            nn.Tanh(),
            nn.Linear(config.rank, n_channels),
        )

    def forward(self, x: torch.Tensor) -> ForwardOutputs:
        batch_size, n_time, _ = x.shape
        z = x.new_zeros(batch_size, self.cfg.rank)
        states = []
        scale = self.cfg.recurrent_scale / (self.cfg.population_dim ** 0.5)
        for t in range(n_time):
            population_drive = (
                self.cfg.input_scale * self.input_to_population(x[:, t, :])
                + self.cfg.recurrent_scale * (z @ self.m_factor.T)
            )
            population = torch.tanh(population_drive)
            proposed_z = scale * (population @ self.n_factor) + self.state_bias
            z = (1.0 - self.cfg.state_leak) * z + self.cfg.state_leak * proposed_z
            states.append(z)
        latents = torch.stack(states, dim=1)
        reconstructed = self.recon_head(latents)
        predicted = self.pred_head(latents)
        return ForwardOutputs(reconstructed=reconstructed, predicted=predicted, latents=latents)


def _compute_reconstruction_losses(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    target_future: torch.Tensor,
    mask: torch.Tensor,
    weights: LossWeights,
) -> Dict[str, torch.Tensor]:
    mask_f = mask.float()
    n_valid = mask_f.sum().clamp(min=1.0)
    recon_err = ((outputs.reconstructed - target_current) ** 2).mean(dim=-1)
    recon_loss = (recon_err * mask_f).sum() / n_valid
    pred_mask = mask_f * weights.future_weight_scale
    pred_err = ((outputs.predicted - target_future) ** 2).mean(dim=-1)
    future_loss = (pred_err * pred_mask).sum() / pred_mask.sum().clamp(min=1.0)
    if target_current.shape[1] > 1:
        recon_diff = outputs.reconstructed[:, 1:, :] - outputs.reconstructed[:, :-1, :]
        target_diff = target_current[:, 1:, :] - target_current[:, :-1, :]
        deriv_err = ((recon_diff - target_diff) ** 2).mean(dim=-1)
        deriv_mask = mask_f[:, 1:]
        derivative_loss = (deriv_err * deriv_mask).sum() / deriv_mask.sum().clamp(min=1.0)
    else:
        derivative_loss = recon_loss.new_zeros(())
    recon_var = outputs.reconstructed.var(dim=1).mean()
    target_var = target_current.var(dim=1).mean()
    variance_loss = (recon_var - target_var).abs()
    return {
        "recon_loss": recon_loss,
        "future_loss": future_loss,
        "derivative_loss": derivative_loss,
        "variance_loss": variance_loss,
    }


def _compute_cpp_shape_prior_losses(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    mask: torch.Tensor,
    times_ms: torch.Tensor,
    weights: LossWeights,
) -> Dict[str, torch.Tensor]:
    zero = target_current.new_zeros(())
    if not weights.enable_cpp_shape_prior:
        return {
            "monotonic_loss": zero,
            "slope_floor_loss": zero,
            "late_amplitude_loss": zero,
            "cpp_mean_alignment_loss": zero,
        }
    recon_cpp = outputs.reconstructed.mean(dim=-1)
    target_cpp = target_current.mean(dim=-1)
    mask_f = mask.float()
    analysis = (times_ms >= weights.analysis_window_ms[0]) & (times_ms <= weights.analysis_window_ms[1])
    late = (times_ms >= weights.late_window_ms[0]) & (times_ms <= weights.late_window_ms[1])
    if analysis.sum() < 2:
        return {
            "monotonic_loss": zero,
            "slope_floor_loss": zero,
            "late_amplitude_loss": zero,
            "cpp_mean_alignment_loss": zero,
        }
    recon_a = recon_cpp[:, analysis]
    target_a = target_cpp[:, analysis]
    mask_a = mask_f[:, analysis]
    recon_slope = recon_a[:, 1:] - recon_a[:, :-1]
    target_slope = target_a[:, 1:] - target_a[:, :-1]
    slope_mask = mask_a[:, 1:]
    monotonic_loss = (torch.relu(-recon_slope) * slope_mask).sum() / slope_mask.sum().clamp(min=1.0)
    floor = weights.slope_floor_ratio * target_slope
    slope_floor_loss = (torch.relu(floor - recon_slope) * slope_mask).sum() / slope_mask.sum().clamp(min=1.0)
    cpp_mean_alignment_loss = (((recon_a - target_a) ** 2) * mask_a).sum() / mask_a.sum().clamp(min=1.0)
    if late.any():
        recon_late = recon_cpp[:, late].mean(dim=1)
        target_late = target_cpp[:, late].mean(dim=1)
        late_amplitude_loss = torch.relu(target_late - recon_late).mean()
    else:
        late_amplitude_loss = zero
    return {
        "monotonic_loss": monotonic_loss,
        "slope_floor_loss": slope_floor_loss,
        "late_amplitude_loss": late_amplitude_loss,
        "cpp_mean_alignment_loss": cpp_mean_alignment_loss,
    }


def _compute_smoothness_loss(outputs: ForwardOutputs) -> torch.Tensor:
    if outputs.latents.shape[1] < 2:
        return outputs.latents.new_zeros(())
    return ((outputs.latents[:, 1:, :] - outputs.latents[:, :-1, :]) ** 2).mean()


def masked_self_supervised_loss(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    target_future: torch.Tensor,
    mask: torch.Tensor,
    times_ms: torch.Tensor,
    weights: LossWeights,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    recon_losses = _compute_reconstruction_losses(outputs, target_current, target_future, mask, weights)
    cpp_proxy = outputs.reconstructed.mean(dim=-1)
    target_proxy = target_current.mean(dim=-1)
    mask_f = mask.float()
    cpp_mean_loss = (((cpp_proxy - target_proxy) ** 2) * mask_f).sum() / mask_f.sum().clamp(min=1.0)
    prior_losses = _compute_cpp_shape_prior_losses(outputs, target_current, mask, times_ms, weights)
    smooth_loss = _compute_smoothness_loss(outputs)
    total_loss = (
        weights.lambda_recon * recon_losses["recon_loss"]
        + weights.lambda_future * recon_losses["future_loss"]
        + weights.lambda_derivative * recon_losses["derivative_loss"]
        + weights.lambda_variance * recon_losses["variance_loss"]
        + weights.lambda_cpp_mean * cpp_mean_loss
        + weights.lambda_cpp_prior
        * (
            weights.lambda_monotonic * prior_losses["monotonic_loss"]
            + weights.lambda_slope_floor * prior_losses["slope_floor_loss"]
            + weights.lambda_late_amplitude * prior_losses["late_amplitude_loss"]
            + weights.lambda_cpp_mean_alignment * prior_losses["cpp_mean_alignment_loss"]
        )
        + weights.lambda_smooth * smooth_loss
    )
    metrics = {
        "total_loss": total_loss.item(),
        "recon_loss": recon_losses["recon_loss"].item(),
        "future_loss": recon_losses["future_loss"].item(),
        "derivative_loss": recon_losses["derivative_loss"].item(),
        "variance_loss": recon_losses["variance_loss"].item(),
        "cpp_mean_loss": cpp_mean_loss.item(),
        "monotonic_loss": prior_losses["monotonic_loss"].item(),
        "slope_floor_loss": prior_losses["slope_floor_loss"].item(),
        "late_amplitude_loss": prior_losses["late_amplitude_loss"].item(),
        "cpp_mean_alignment_loss": prior_losses["cpp_mean_alignment_loss"].item(),
        "smooth_loss": smooth_loss.item(),
    }
    return total_loss, metrics


def low_rank_self_supervised_loss(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    target_future: torch.Tensor,
    mask: torch.Tensor,
    times_ms: torch.Tensor,
    weights: LossWeights,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    return masked_self_supervised_loss(outputs, target_current, target_future, mask, times_ms, weights)

## 6 · Training and Testing Pipeline

In [ ]:
def _extract_checkpoint_state_dict(ckpt: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    for key in ("model_state_dict", "model_state"):
        if key in ckpt:
            return ckpt[key]
    raise KeyError("Checkpoint is missing model weights.")


def _load_checkpoint_weights(model: nn.Module, ckpt: Dict[str, Any]) -> None:
    model.load_state_dict(_extract_checkpoint_state_dict(ckpt), strict=True)


def _run_epoch(
    model: CPPLowRankRNN,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    config: TrainingConfig,
    device: torch.device,
    *,
    train: bool,
) -> Dict[str, float]:
    model.train(train)
    rows: List[Dict[str, float]] = []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x = batch["eeg"].to(device)
            x_future = batch["target_future"].to(device)
            mask = batch["mask"].to(device)
            times_ms = batch["times_ms"][0].to(device)
            outputs = model(x)
            loss, metrics = low_rank_self_supervised_loss(outputs, x, x_future, mask, times_ms, config.loss)
            if train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                optimizer.step()
            rows.append(metrics)
    return aggregate_metric_dicts(rows)


def train_one_epoch(model, loader, optimizer, config, device) -> Dict[str, float]:
    return _run_epoch(model, loader, optimizer, config, device, train=True)


def evaluate_one_epoch(model, loader, config, device) -> Dict[str, float]:
    return _run_epoch(model, loader, None, config, device, train=False)


def _save_loss_curves(history: pd.DataFrame, output_dir: Path) -> None:
    ensure_dir(output_dir)
    fig, ax = plt.subplots(figsize=(3.5, 2.4))
    ax.plot(history["epoch"], history["train_total_loss"], label="train", color=NATURE_BLUE, linewidth=1.4)
    ax.plot(history["epoch"], history["val_total_loss"], label="validation", color=NATURE_ORANGE, linewidth=1.4)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Total loss")
    ax.set_title("Rank=5 low-rank RNN training curve")
    clean_axes(ax)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "training_validation_loss_curve")
    plt.close(fig)


def _collect_one_batch(model: CPPLowRankRNN, loader: DataLoader, device: torch.device) -> Tuple[Dict[str, torch.Tensor], ForwardOutputs]:
    model.eval()
    batch = next(iter(loader))
    x = batch["eeg"].to(device)
    with torch.no_grad():
        outputs = model(x)
    return batch, ForwardOutputs(
        reconstructed=outputs.reconstructed.cpu(),
        predicted=outputs.predicted.cpu(),
        latents=outputs.latents.cpu(),
    )


def _save_reconstruction_examples(model: CPPLowRankRNN, loader: DataLoader, output_dir: Path, device: torch.device) -> None:
    ensure_dir(output_dir)
    batch, outputs = _collect_one_batch(model, loader, device)
    x = batch["eeg"].cpu().numpy()
    recon = outputs.reconstructed.numpy()
    pred = outputs.predicted.numpy()
    times = batch["times_ms"][0].cpu().numpy()
    channels = ["CP1", "CP2", "CPz"]
    trial = 0
    fig, axes = plt.subplots(3, 1, figsize=(4.2, 4.2), sharex=True)
    for ch, ax in enumerate(axes):
        ax.plot(times, x[trial, :, ch], label="observed", color=NATURE_DARK, linewidth=1.1)
        ax.plot(times, recon[trial, :, ch], label="reconstructed", color=NATURE_BLUE, linewidth=1.1)
        ax.plot(times, pred[trial, :, ch], label="future prediction", color=NATURE_TEAL, linewidth=1.0, alpha=0.85)
        ax.set_ylabel(channels[ch])
        ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.45)
        clean_axes(ax)
    axes[-1].set_xlabel("Time from response (ms)")
    axes[0].legend(loc="upper left", fontsize=7, frameon=False)
    fig.suptitle("Rank=5 low-rank RNN reconstruction and future prediction")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "reconstruction_and_prediction_examples")
    plt.close(fig)


def _save_cpp_average_plot(model: CPPLowRankRNN, loader: DataLoader, output_dir: Path, device: torch.device) -> None:
    ensure_dir(output_dir)
    observed_cpp = []
    recon_cpp = []
    pred_cpp = []
    times = None
    model.eval()
    with torch.no_grad():
        for batch in loader:
            x = batch["eeg"].to(device)
            out = model(x)
            observed_cpp.append(x.mean(dim=-1).cpu().numpy())
            recon_cpp.append(out.reconstructed.mean(dim=-1).cpu().numpy())
            pred_cpp.append(out.predicted.mean(dim=-1).cpu().numpy())
            times = batch["times_ms"][0].cpu().numpy()
    obs = np.concatenate(observed_cpp).mean(axis=0)
    rec = np.concatenate(recon_cpp).mean(axis=0)
    pred = np.concatenate(pred_cpp).mean(axis=0)
    fig, ax = plt.subplots(figsize=(3.8, 2.5))
    ax.plot(times, obs, label="observed CPP", color=NATURE_DARK, linewidth=1.5)
    ax.plot(times, rec, label="reconstructed CPP", color=NATURE_BLUE, linewidth=1.5)
    ax.plot(times, pred, label="future-predicted CPP", color=NATURE_TEAL, linewidth=1.2, alpha=0.85)
    ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.45)
    ax.set_xlabel("Time from response (ms)")
    ax.set_ylabel("Normalised CPP proxy")
    ax.set_title("CPP average comparison")
    clean_axes(ax)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "cpp_average_reconstruction_comparison")
    plt.close(fig)


def _save_rank5_latent_preview(model: CPPLowRankRNN, loader: DataLoader, output_dir: Path, device: torch.device) -> None:
    ensure_dir(output_dir)
    all_z = []
    times = None
    model.eval()
    with torch.no_grad():
        for batch in loader:
            x = batch["eeg"].to(device)
            out = model(x)
            all_z.append(out.latents.cpu().numpy())
            times = batch["times_ms"][0].cpu().numpy()
    latents = np.concatenate(all_z, axis=0)
    mean_z = latents.mean(axis=0)
    fig, ax = plt.subplots(figsize=(4.5, 2.8))
    for k in range(mean_z.shape[-1]):
        ax.plot(times, mean_z[:, k], label=f"z{k + 1}(t)", color=NATURE_PALETTE[k], linewidth=1.2)
    ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.45)
    ax.set_xlabel("Time from response (ms)")
    ax.set_ylabel("Latent value")
    ax.set_title("Rank=5 low-rank latent preview: z1–z5")
    clean_axes(ax)
    ax.legend(ncol=5, fontsize=7, frameon=False)
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "rank5_latent_preview_z1_to_z5")
    plt.close(fig)

    fig, axes = plt.subplots(5, 1, figsize=(4.5, 6.0), sharex=True)
    for k, ax in enumerate(axes):
        for trial in range(min(12, latents.shape[0])):
            ax.plot(times, latents[trial, :, k], color=NATURE_PALETTE[k], alpha=0.35, linewidth=0.7)
        ax.set_ylabel(f"z{k + 1}")
        ax.axvline(0, color=NATURE_DARK, linewidth=0.7, alpha=0.4)
        clean_axes(ax)
    axes[-1].set_xlabel("Time from response (ms)")
    fig.suptitle("Trial-level rank-5 latent examples")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "trial_level_rank5_latent_examples")
    plt.close(fig)


def train_low_rank_r5_model(dataset_dir: Path, output_dir: Path, config: TrainingConfig) -> Dict[str, object]:
    config = _coerce_training_config(config)
    if config.model.rank != 5:
        raise ValueError("Training is fixed to rank=5.")
    set_seed(config.seed)
    ensure_dir(output_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    train_loader, val_loader, test_loader, split_indices = make_dataloaders(eeg, targets, mask, times_ms, config)
    model = CPPLowRankRNN(n_channels=eeg.shape[-1], config=config.model).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    best_val_loss = float("inf")
    patience = 0
    history_rows: List[Dict[str, float]] = []
    checkpoint_path = output_dir / "best_low_rank_r5_model.pt"
    start_time = time.time()

    for epoch in range(config.max_epochs):
        train_metrics = train_one_epoch(model, train_loader, optimizer, config, device)
        val_metrics = evaluate_one_epoch(model, val_loader, config, device)
        row = {"epoch": epoch + 1}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history_rows.append(row)
        print(
            f"Epoch {epoch + 1:03d}/{config.max_epochs} | "
            f"train={train_metrics['total_loss']:.5f} | val={val_metrics['total_loss']:.5f}"
        )
        if val_metrics["total_loss"] < best_val_loss:
            best_val_loss = val_metrics["total_loss"]
            patience = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "epoch": epoch,
                    "val_loss": best_val_loss,
                    "split_indices": split_indices,
                    "rank": 5,
                },
                checkpoint_path,
            )
        else:
            patience += 1
            if patience >= config.early_stopping_patience:
                print(f"Early stopping after {epoch + 1} epochs.")
                break

    history = pd.DataFrame(history_rows)
    save_dataframe(output_dir / "training_history.csv", history)
    _save_loss_curves(history, output_dir)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    _load_checkpoint_weights(model, ckpt)
    _save_reconstruction_examples(model, val_loader, output_dir, device)
    _save_cpp_average_plot(model, val_loader, output_dir, device)
    _save_rank5_latent_preview(model, val_loader, output_dir, device)
    return {
        "best_val_loss": best_val_loss,
        "checkpoint_path": checkpoint_path,
        "n_epochs_trained": len(history_rows),
        "elapsed_seconds": time.time() - start_time,
        "split_indices": split_indices,
    }


def test_low_rank_r5_model(
    checkpoint_path: Path,
    dataset_dir: Path,
    output_json: Path,
) -> Dict[str, float]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = _coerce_training_config(ckpt["config"])
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    train_loader, val_loader, test_loader, split_indices = make_dataloaders(eeg, targets, mask, times_ms, config)
    model = CPPLowRankRNN(n_channels=eeg.shape[-1], config=config.model).to(device)
    _load_checkpoint_weights(model, ckpt)
    metrics = evaluate_one_epoch(model, test_loader, config, device)
    prefixed = {f"test_{key}": value for key, value in metrics.items()}
    write_json(output_json, prefixed)
    print("Rank=5 low-rank RNN test results:")
    for key, value in prefixed.items():
        print(f"{key} = {value:.6f}")
    return prefixed

## 7 · ▶ Run Low-Rank Rank=5 Training

In [ ]:
_cfg = TrainingConfig(
    seed=42,
    model=LowRankRNNConfig(
        rank=5,
        population_dim=64,
        input_scale=1.0,
        recurrent_scale=1.0,
        state_leak=0.25,
    ),
    loss=LossWeights(
        lambda_recon=1.0,
        lambda_future=0.2,
        lambda_cpp_prior=0.1,
        enable_cpp_shape_prior=True,
    ),
)

_train_result = train_low_rank_r5_model(
    dataset_dir=DATASET_DIR,
    output_dir=LOW_RANK_R5_CHECKPOINT_DIR,
    config=_cfg,
)
print("Training complete.")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in _train_result.items() if k != "split_indices"}, indent=2))

## 8 · ▶ Test Best Low-Rank Rank=5 Model

In [ ]:
test_metrics = test_low_rank_r5_model(
    checkpoint_path=LOW_RANK_R5_BEST_CKPT,
    dataset_dir=DATASET_DIR,
    output_json=LOW_RANK_R5_TEST_METRICS,
)
test_metrics

## 9 · ▶ Export Rank=5 Latent States

In [ ]:
def export_low_rank_r5_latents_from_checkpoint(
    checkpoint_path: Path,
    dataset_dir: Path,
    output_path: Path,
) -> Path:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = _coerce_training_config(ckpt["config"])
    if config.model.rank != 5:
        raise ValueError("Expected rank=5 checkpoint.")
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    model = CPPLowRankRNN(n_channels=eeg.shape[-1], config=config.model).to(device)
    _load_checkpoint_weights(model, ckpt)
    model.eval()
    all_latents: List[np.ndarray] = []
    all_trial_ids: List[np.ndarray] = []
    batch_size = 256
    eeg_tensor = torch.as_tensor(eeg, dtype=torch.float32)
    with torch.no_grad():
        for start in range(0, len(eeg), batch_size):
            x_batch = eeg_tensor[start : start + batch_size].to(device)
            out = model(x_batch)
            all_latents.append(out.latents.cpu().numpy())
            all_trial_ids.append(np.arange(start, min(start + batch_size, len(eeg))))
    latents = np.concatenate(all_latents, axis=0).astype(np.float32)
    trial_ids = metadata["trial_id"].to_numpy() if "trial_id" in metadata.columns else np.concatenate(all_trial_ids)
    ensure_dir(output_path.parent)
    np.savez_compressed(
        output_path,
        latents=latents,
        times_ms=times_ms.astype(np.float32),
        trial_ids=trial_ids,
        rank=np.array(5),
        model_type=np.array("CPPLowRankRNN_rank5"),
        checkpoint_path=np.array(str(checkpoint_path)),
    )
    print(f"Saved Rank=5 latents: {output_path}")
    print(f"latents shape: {latents.shape}")
    return output_path


_latent_path = export_low_rank_r5_latents_from_checkpoint(
    checkpoint_path=LOW_RANK_R5_BEST_CKPT,
    dataset_dir=DATASET_DIR,
    output_path=LOW_RANK_R5_LATENT_PATH,
)
_z_preview = np.load(_latent_path, allow_pickle=True)
print("Available keys:", sorted(_z_preview.files))
print("Rank=5 latent shape:", _z_preview["latents"].shape)

## 10 · Low-Rank Rank=5 Latent Diagnostics

This section visualises z1–z5 and reconstruction quality. The z variables are the Rank=5 CPPLowRankRNN recurrent state, with more capacity than rank-3 while still exposing a compact interpretable state.

In [ ]:
def run_low_rank_r5_diagnostics(
    checkpoint_path: Path,
    dataset_dir: Path,
    latent_path: Path,
    output_dir: Path,
) -> Dict[str, str]:
    ensure_dir(output_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = _coerce_training_config(ckpt["config"])
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    train_loader, val_loader, test_loader, split_indices = make_dataloaders(eeg, targets, mask, times_ms, config)
    model = CPPLowRankRNN(n_channels=eeg.shape[-1], config=config.model).to(device)
    _load_checkpoint_weights(model, ckpt)
    _save_reconstruction_examples(model, test_loader, output_dir, device)
    _save_cpp_average_plot(model, test_loader, output_dir, device)
    _save_rank5_latent_preview(model, test_loader, output_dir, device)

    z = np.load(latent_path, allow_pickle=True)
    latents = z["latents"]
    times = z["times_ms"]
    mean_z = latents.mean(axis=0)
    std_z = latents.std(axis=0)
    fig, axes = plt.subplots(5, 1, figsize=(4.5, 6.2), sharex=True)
    for k, ax in enumerate(axes):
        ax.plot(times, mean_z[:, k], color=NATURE_PALETTE[k], label=f"z{k + 1}(t)", linewidth=1.2)
        ax.fill_between(times, mean_z[:, k] - std_z[:, k], mean_z[:, k] + std_z[:, k], color=NATURE_PALETTE[k], alpha=0.12, linewidth=0)
        ax.axvline(0, color=NATURE_DARK, linewidth=0.7, alpha=0.45)
        ax.set_ylabel(f"z{k + 1}")
        clean_axes(ax)
    axes[-1].set_xlabel("Time from response (ms)")
    fig.suptitle("z1–z5 average trajectories with across-trial spread")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "rank5_z1_to_z5_average_trajectories")
    plt.close(fig)

    log_rt = np.log(metadata["RT_ms"].values.astype(float))
    rt_corr_rows = []
    corr_mat = np.zeros((latents.shape[-1], latents.shape[1]), dtype=np.float32)
    for k in range(latents.shape[-1]):
        for t_idx, t_ms in enumerate(times):
            corr = np.corrcoef(latents[:, t_idx, k], log_rt)[0, 1]
            corr_mat[k, t_idx] = corr
            rt_corr_rows.append({"time_ms": float(t_ms), "latent": f"z{k + 1}", "correlation_with_log_rt": float(corr)})
    rt_corr_df = pd.DataFrame(rt_corr_rows)
    save_dataframe(output_dir / "z_rt_time_resolved_correlation.csv", rt_corr_df)
    fig, ax = plt.subplots(figsize=(4.6, 2.8))
    for k in range(latents.shape[-1]):
        ax.plot(times, corr_mat[k], color=NATURE_PALETTE[k], linewidth=1.2, label=f"z{k + 1}")
    ax.axhline(0, color=NATURE_LIGHT_GRAY, linewidth=0.8)
    ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.45)
    ax.set_xlabel("Time from response (ms)")
    ax.set_ylabel("Correlation with log RT")
    ax.set_title("Time-resolved z–RT correlation")
    clean_axes(ax)
    ax.legend(ncol=5, frameon=False, fontsize=7)
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "z_rt_time_resolved_correlation")
    plt.close(fig)

    rt_groups = pd.qcut(metadata["RT_ms"], q=3, labels=["fast", "medium", "slow"])
    group_colors = {"fast": NATURE_BLUE, "medium": NATURE_GRAY, "slow": NATURE_RED}
    fig, axes = plt.subplots(5, 1, figsize=(4.6, 6.4), sharex=True)
    trajectory_rows = []
    for k, ax in enumerate(axes):
        for label in ["fast", "medium", "slow"]:
            group_mask = (rt_groups == label).to_numpy()
            group_mean = latents[group_mask, :, k].mean(axis=0)
            group_sem = latents[group_mask, :, k].std(axis=0) / np.sqrt(group_mask.sum())
            ax.plot(times, group_mean, color=group_colors[label], linewidth=1.2, label=label)
            ax.fill_between(times, group_mean - group_sem, group_mean + group_sem, color=group_colors[label], alpha=0.12, linewidth=0)
            for t_ms, value in zip(times, group_mean):
                trajectory_rows.append({"latent": f"z{k + 1}", "rt_group": label, "time_ms": float(t_ms), "mean": float(value)})
        ax.axvline(0, color=NATURE_DARK, linewidth=0.7, alpha=0.45)
        ax.set_ylabel(f"z{k + 1}")
        clean_axes(ax)
    axes[0].legend(frameon=False, ncol=3, fontsize=7)
    axes[-1].set_xlabel("Time from response (ms)")
    fig.suptitle("Rank-5 latent trajectories by RT tertile")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "z_trajectories_by_rt_tertile")
    plt.close(fig)
    save_dataframe(output_dir / "z_trajectories_by_rt_tertile.csv", pd.DataFrame(trajectory_rows))

    cpp = eeg.mean(axis=-1)
    late_mask = (times_ms >= -120.0) & (times_ms <= -50.0)
    slope_mask = (times_ms >= -300.0) & (times_ms <= -50.0)
    cpp_amplitude = cpp.mean(axis=1)
    late_cpp_amplitude = cpp[:, late_mask].mean(axis=1) if late_mask.any() else np.zeros(len(metadata))
    if slope_mask.sum() >= 2:
        t_slope = times_ms[slope_mask]
        t_slope = (t_slope - t_slope.mean()) / (t_slope.std() if t_slope.std() > 0 else 1.0)
        cpp_slope = np.array([np.polyfit(t_slope, cpp[i, slope_mask], 1)[0] for i in range(cpp.shape[0])])
    else:
        cpp_slope = np.zeros(len(metadata))
    full_window = (times >= -600.0) & (times <= -50.0)
    z_features = {f"z{k + 1}_mean": latents[:, full_window, k].mean(axis=1) for k in range(5)}
    feature_df = pd.DataFrame(z_features)
    feature_df["CPP amplitude"] = cpp_amplitude
    feature_df["CPP slope"] = cpp_slope
    feature_df["Late CPP amplitude"] = late_cpp_amplitude
    feature_df["RT"] = metadata["RT_ms"].values.astype(float)
    for optional_col in ["condition", "difficulty", "evidence_strength", "correctness"]:
        if optional_col in metadata.columns:
            series = metadata[optional_col]
            feature_df[optional_col] = series.astype("category").cat.codes if not np.issubdtype(series.dtype, np.number) else series
    corr_df = feature_df.corr(numeric_only=True).loc[[f"z{k + 1}_mean" for k in range(5)]]
    save_dataframe(output_dir / "z_cpp_behavior_correlation_matrix.csv", corr_df.reset_index(names="latent"))
    fig, ax = plt.subplots(figsize=(5.2, 2.6))
    image = ax.imshow(corr_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_yticks(np.arange(corr_df.shape[0]))
    ax.set_yticklabels([f"z{k + 1}" for k in range(5)])
    ax.set_xticks(np.arange(corr_df.shape[1]))
    ax.set_xticklabels(corr_df.columns, rotation=35, ha="right")
    ax.set_title("z, CPP, and behaviour correlations")
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(image, ax=ax, shrink=0.8)
    cbar.set_label("Pearson r")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "z_cpp_behavior_correlation_heatmap")
    plt.close(fig)

    summary = {
        "diagnostics_dir": str(output_dir),
        "latent_shape": list(latents.shape),
        "rank": 5,
        "figures": sorted(path.name for path in output_dir.glob("*.png")),
        "tables": sorted(path.name for path in output_dir.glob("*.csv")),
    }
    write_json(output_dir / "diagnostics_summary.json", summary)
    return summary


diagnostics_summary = run_low_rank_r5_diagnostics(
    checkpoint_path=LOW_RANK_R5_BEST_CKPT,
    dataset_dir=DATASET_DIR,
    latent_path=LOW_RANK_R5_LATENT_PATH,
    output_dir=LOW_RANK_R5_DIAGNOSTICS_DIR,
)
diagnostics_summary

## 11 · ▶ Ridge Regression with Rank=5 Latents

In [ ]:
_ALPHA_GRID = np.logspace(-2, 7, 80)


def _load_latents(latent_npz: Path) -> Tuple[np.ndarray, np.ndarray]:
    payload = np.load(latent_npz, allow_pickle=True)
    latents = payload["latents"].astype(np.float32)
    times_ms = payload["times_ms"].astype(np.float32)
    if latents.shape[-1] != 5:
        raise ValueError(f"Expected rank-5 latents, got shape {latents.shape}")
    return latents, times_ms


def _load_behaviour(dataset_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(dataset_dir / "metadata.csv")
    if "RT_ms" not in df.columns:
        raise ValueError("metadata.csv must contain RT_ms")
    if "subj_idx" not in df.columns and "subject_id" not in df.columns:
        raise ValueError("metadata.csv must contain subj_idx or subject_id for subject-aware cross-validation")
    return df


def _subject_groups(df: pd.DataFrame) -> np.ndarray:
    group_col = "subj_idx" if "subj_idx" in df.columns else "subject_id"
    return df[group_col].values


def _window_features(latents: np.ndarray, times_ms: np.ndarray, window_ms: Tuple[float, float]) -> np.ndarray:
    mask = (times_ms >= window_ms[0]) & (times_ms <= window_ms[1])
    if not mask.any():
        raise ValueError(f"Window {window_ms} ms contains no time steps.")
    return latents[:, mask, :].mean(axis=1).astype(np.float32)


def _baseline_design(df: pd.DataFrame) -> np.ndarray:
    parts: List[np.ndarray] = []
    diff_col = next((c for c in ("coherence", "difficulty", "evidence_strength") if c in df.columns), None)
    if diff_col is not None:
        vals = df[diff_col].astype("category").cat.codes.values.astype(np.float32) if not np.issubdtype(df[diff_col].dtype, np.number) else df[diff_col].values.astype(np.float32)
        std = vals.std()
        parts.append(((vals - vals.mean()) / (std if std > 0 else 1.0)).reshape(-1, 1))
    else:
        parts.append(np.zeros((len(df), 1), dtype=np.float32))
    if "correctness" in df.columns:
        vals = df["correctness"].values
        if not np.issubdtype(df["correctness"].dtype, np.number):
            vals = pd.Series(vals).astype("category").cat.codes.values
        parts.append(vals.astype(np.float32).reshape(-1, 1))
    if "condition" in df.columns:
        condition_codes = pd.get_dummies(df["condition"], drop_first=True).values.astype(np.float32)
        if condition_codes.size:
            parts.append(condition_codes)
    return np.concatenate(parts, axis=1)


def _cpp_features(dataset_dir: Path, df: pd.DataFrame) -> Optional[np.ndarray]:
    eeg_path = dataset_dir / "eeg_cpp_trials.npy"
    times_path = dataset_dir / "times_ms.npy"
    if not eeg_path.exists() or not times_path.exists():
        return None
    eeg = np.load(eeg_path).astype(np.float32)
    if eeg.shape[0] != len(df):
        return None
    times_ms = np.load(times_path).astype(np.float32)
    cpp = eeg.mean(axis=-1)
    late = (times_ms >= -120.0) & (times_ms <= -50.0)
    slope_window = (times_ms >= -300.0) & (times_ms <= -50.0)
    late_amp = cpp[:, late].mean(axis=1) if late.any() else np.zeros(len(df))
    if slope_window.sum() >= 2:
        t = times_ms[slope_window]
        t = (t - t.mean()) / (t.std() if t.std() > 0 else 1.0)
        slopes = np.array([np.polyfit(t, cpp[i, slope_window], 1)[0] for i in range(len(df))])
    else:
        slopes = np.zeros(len(df))
    mean_amp = cpp.mean(axis=1)
    return np.stack([mean_amp, late_amp, slopes], axis=1).astype(np.float32)


def _model_cv_predictions(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    splits: List[Tuple[np.ndarray, np.ndarray]],
    model_name: str,
    window: str,
    feature_names: Optional[List[str]] = None,
    scale: bool = True,
) -> Tuple[np.ndarray, List[Dict[str, float | str]], List[Dict[str, float | str]]]:
    y_pred = np.empty_like(y, dtype=np.float64)
    fold_rows: List[Dict[str, float | str]] = []
    coefficient_rows: List[Dict[str, float | str]] = []
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx], y[test_idx]
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_te = scaler.transform(X_te)
        inner_groups = groups[train_idx]
        inner_unique = np.unique(inner_groups)
        if len(inner_unique) >= 2:
            inner_cv = list(GroupKFold(n_splits=min(5, len(inner_unique))).split(X_tr, y_tr, groups=inner_groups))
        else:
            inner_cv = 5
        ridge_cv = RidgeCV(alphas=_ALPHA_GRID, cv=inner_cv)
        ridge_cv.fit(X_tr, y_tr)
        model = Ridge(alpha=float(ridge_cv.alpha_), solver="svd")
        model.fit(X_tr, y_tr)
        fold_pred = model.predict(X_te)
        y_pred[test_idx] = fold_pred
        fold_rows.append(
            {
                "window": window,
                "model": model_name,
                "fold": fold,
                "r2": float(r2_score(y_te, fold_pred)),
                "corr": float(np.corrcoef(y_te, fold_pred)[0, 1]),
                "alpha": float(ridge_cv.alpha_),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
            }
        )
        names = feature_names or [f"feature_{i}" for i in range(X.shape[1])]
        for name, value in zip(names, model.coef_):
            coefficient_rows.append(
                {
                    "window": window,
                    "model": model_name,
                    "fold": fold,
                    "feature": name,
                    "coefficient": float(value),
                }
            )
    return y_pred, fold_rows, coefficient_rows


def _prediction_r2(y_true: np.ndarray, y_pred: np.ndarray, indices: np.ndarray) -> float:
    return float(r2_score(y_true[indices], y_pred[indices]))


def _save_ridge_delta_forestplot(ci_df: pd.DataFrame, output_dir: Path) -> None:
    contrast_order = ["baseline+z - baseline", "baseline+cpp+z - baseline+cpp", "baseline+shuffled-z - baseline"]
    colors = {
        "baseline+z - baseline": NATURE_TEAL,
        "baseline+cpp+z - baseline+cpp": NATURE_ORANGE,
        "baseline+shuffled-z - baseline": NATURE_GRAY,
    }
    markers = {
        "baseline+z - baseline": "o",
        "baseline+cpp+z - baseline+cpp": "s",
        "baseline+shuffled-z - baseline": "o",
    }
    windows = list(dict.fromkeys(ci_df["window"].tolist()))
    fig, ax = plt.subplots(figsize=(4.8, 2.9))
    offsets = {-1: -0.18, 0: 0.0, 1: 0.18}
    for c_idx, contrast in enumerate(contrast_order):
        subset = ci_df[ci_df["contrast"] == contrast]
        for w_idx, window in enumerate(windows):
            row = subset[subset["window"] == window]
            if row.empty:
                continue
            row = row.iloc[0]
            y = w_idx + offsets[c_idx - 1]
            ax.errorbar(
                row["mean_delta_r2"],
                y,
                xerr=[[row["mean_delta_r2"] - row["ci_lower"]], [row["ci_upper"] - row["mean_delta_r2"]]],
                fmt=markers[contrast],
                color=colors[contrast],
                markerfacecolor="white" if "shuffled" in contrast else colors[contrast],
                markeredgecolor=colors[contrast],
                elinewidth=1.0,
                capsize=2,
                markersize=4,
                label=contrast if w_idx == 0 else None,
            )
    ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.6)
    ax.set_yticks(np.arange(len(windows)))
    ax.set_yticklabels(windows)
    ax.invert_yaxis()
    ax.set_xlabel("Incremental cross-validated R², ΔR²")
    ax.set_ylabel("Time window")
    ax.set_title("Incremental RT prediction from rank-5 z variables")
    clean_axes(ax)
    ax.legend(frameon=False, fontsize=7, loc="best")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "ridge_delta_r2_forestplot")
    plt.close(fig)


def _save_raw_r2_dot_plot(fold_df: pd.DataFrame, output_dir: Path) -> None:
    model_order = ["baseline", "z-only", "baseline+z", "baseline+cpp", "baseline+cpp+z", "baseline+shuffled-z"]
    windows = list(dict.fromkeys(fold_df["window"].tolist()))
    fig, axes = plt.subplots(1, len(windows), figsize=(8.0, 2.8), sharey=True)
    if len(windows) == 1:
        axes = [axes]
    for ax, window in zip(axes, windows):
        sub = fold_df[fold_df["window"] == window]
        for i, model in enumerate(model_order):
            vals = sub[sub["model"] == model]["r2"].values
            if len(vals) == 0:
                continue
            jitter = np.linspace(-0.06, 0.06, len(vals)) if len(vals) > 1 else np.array([0.0])
            ax.scatter(np.full(len(vals), i) + jitter, vals, s=12, color=NATURE_BLUE if "z" in model else NATURE_GRAY, alpha=0.75, linewidth=0)
            ax.plot(i, vals.mean(), marker="_", color=NATURE_DARK, markersize=10)
        ax.set_title(window)
        ax.set_xticks(range(len(model_order)))
        ax.set_xticklabels(model_order, rotation=45, ha="right")
        clean_axes(ax)
    axes[0].set_ylabel("Fold-level R²")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "ridge_raw_r2_dotplot")
    plt.close(fig)


def _save_paired_improvement_plot(fold_df: pd.DataFrame, output_dir: Path) -> None:
    pairs = [("baseline", "baseline+z"), ("baseline+cpp", "baseline+cpp+z")]
    windows = list(dict.fromkeys(fold_df["window"].tolist()))
    fig, axes = plt.subplots(1, len(windows), figsize=(7.4, 2.7), sharey=True)
    if len(windows) == 1:
        axes = [axes]
    for ax, window in zip(axes, windows):
        sub = fold_df[fold_df["window"] == window]
        x_pos = 0
        for left, right in pairs:
            left_vals = sub[sub["model"] == left].sort_values("fold")
            right_vals = sub[sub["model"] == right].sort_values("fold")
            if left_vals.empty or right_vals.empty:
                continue
            for _, lrow in left_vals.iterrows():
                rrow = right_vals[right_vals["fold"] == lrow["fold"]]
                if rrow.empty:
                    continue
                ax.plot([x_pos, x_pos + 1], [lrow["r2"], rrow.iloc[0]["r2"]], color=NATURE_GRAY, linewidth=0.7, alpha=0.7)
            ax.scatter([x_pos] * len(left_vals), left_vals["r2"], color=NATURE_GRAY, s=12)
            ax.scatter([x_pos + 1] * len(right_vals), right_vals["r2"], color=NATURE_TEAL if right.endswith("z") else NATURE_ORANGE, s=12)
            x_pos += 3
        ax.set_xticks([0, 1, 3, 4])
        ax.set_xticklabels(["base", "+z", "cpp", "cpp+z"], rotation=30, ha="right")
        ax.set_title(window)
        clean_axes(ax)
    axes[0].set_ylabel("Fold-level R²")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "ridge_paired_improvement_plot")
    plt.close(fig)


def _save_shuffled_control_plot(fold_df: pd.DataFrame, output_dir: Path) -> None:
    model_order = ["baseline", "baseline+z", "baseline+shuffled-z"]
    windows = list(dict.fromkeys(fold_df["window"].tolist()))
    fig, axes = plt.subplots(1, len(windows), figsize=(6.8, 2.5), sharey=True)
    if len(windows) == 1:
        axes = [axes]
    for ax, window in zip(axes, windows):
        sub = fold_df[fold_df["window"] == window]
        for i, model in enumerate(model_order):
            vals = sub[sub["model"] == model]["r2"].values
            if len(vals) == 0:
                continue
            color = NATURE_TEAL if model == "baseline+z" else NATURE_GRAY
            face = "white" if "shuffled" in model else color
            ax.scatter(np.full(len(vals), i), vals, s=18, facecolor=face, edgecolor=color, linewidth=0.8)
            ax.plot(i, vals.mean(), marker="_", color=NATURE_DARK, markersize=10)
        ax.set_xticks(range(len(model_order)))
        ax.set_xticklabels(["base", "+z", "+shuffled z"], rotation=30, ha="right")
        ax.set_title(window)
        clean_axes(ax)
    axes[0].set_ylabel("Fold-level R²")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "ridge_shuffled_control_plot")
    plt.close(fig)


def _save_z_coefficient_forestplot(coef_df: pd.DataFrame, output_dir: Path) -> None:
    z_coef = coef_df[(coef_df["model"].isin(["baseline+z", "baseline+cpp+z", "z-only"])) & (coef_df["feature"].str.match(r"z[1-5]_mean"))]
    if z_coef.empty:
        return
    rows = []
    for (window, model, feature), group in z_coef.groupby(["window", "model", "feature"]):
        mean, lo, hi = bootstrap_ci(group["coefficient"].values, seed=42)
        rows.append({"window": window, "model": model, "feature": feature, "mean": mean, "ci_lower": lo, "ci_upper": hi})
    coef_summary = pd.DataFrame(rows)
    save_dataframe(output_dir / "ridge_z_standardized_coefficient_ci.csv", coef_summary)
    windows = list(dict.fromkeys(coef_summary["window"].tolist()))
    fig, axes = plt.subplots(1, len(windows), figsize=(8.0, 2.8), sharex=True, sharey=True)
    if len(windows) == 1:
        axes = [axes]
    feature_order = [f"z{k}_mean" for k in range(1, 6)]
    for ax, window in zip(axes, windows):
        sub = coef_summary[(coef_summary["window"] == window) & (coef_summary["model"] == "baseline+z")]
        for y_idx, feature in enumerate(feature_order):
            row = sub[sub["feature"] == feature]
            if row.empty:
                continue
            row = row.iloc[0]
            ax.errorbar(row["mean"], y_idx, xerr=[[row["mean"] - row["ci_lower"]], [row["ci_upper"] - row["mean"]]], fmt="o", color=NATURE_BLUE, elinewidth=1.0, capsize=2, markersize=4)
        ax.axvline(0, color=NATURE_DARK, linewidth=0.8, alpha=0.6)
        ax.set_yticks(range(5))
        ax.set_yticklabels([f"z{k}" for k in range(1, 6)])
        ax.invert_yaxis()
        ax.set_title(window)
        clean_axes(ax)
    axes[0].set_ylabel("Latent variable")
    fig.supxlabel("Standardized coefficient")
    fig.tight_layout()
    save_publication_figure(fig, output_dir / "ridge_z_standardized_coefficient_forestplot")
    plt.close(fig)


def _permutation_test_from_predictions(
    y: np.ndarray,
    pred_base: np.ndarray,
    pred_aug: np.ndarray,
    splits: List[Tuple[np.ndarray, np.ndarray]],
    n_perm: int,
    seed: int,
) -> Tuple[float, float]:
    observed = np.mean([_prediction_r2(y, pred_aug, test_idx) - _prediction_r2(y, pred_base, test_idx) for _, test_idx in splits])
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        fold_deltas = []
        for _, test_idx in splits:
            shuffled = pred_aug.copy()
            shuffled[test_idx] = rng.permutation(shuffled[test_idx])
            fold_deltas.append(_prediction_r2(y, shuffled, test_idx) - _prediction_r2(y, pred_base, test_idx))
        null.append(np.mean(fold_deltas))
    p_value = (np.sum(np.asarray(null) >= observed) + 1.0) / (n_perm + 1.0)
    return float(observed), float(p_value)


def run_low_rank_r5_ridge_rt_analysis(
    latent_npz: Path,
    dataset_dir: Path,
    output_dir: Path,
    window_definitions: Optional[Dict[str, Tuple[float, float]]] = None,
    n_outer_folds: int = 5,
    n_perm: int = 1000,
) -> Dict[str, Any]:
    ensure_dir(output_dir)
    if window_definitions is None:
        window_definitions = {
            "early (-600 to -300 ms)": (-600.0, -300.0),
            "mid (-300 to -120 ms)": (-300.0, -120.0),
            "late (-120 to -50 ms)": (-120.0, -50.0),
            "full (-600 to -50 ms)": (-600.0, -50.0),
        }
    latents, times_ms = _load_latents(latent_npz)
    df = _load_behaviour(dataset_dir)
    if len(df) != latents.shape[0]:
        raise ValueError(f"latents has {latents.shape[0]} trials but metadata has {len(df)} rows")
    log_rt = np.log(df["RT_ms"].values.astype(np.float64))
    groups = _subject_groups(df)
    n_splits = min(n_outer_folds, len(np.unique(groups)))
    splits = list(GroupKFold(n_splits=n_splits).split(np.zeros(len(df)), log_rt, groups=groups))
    X_baseline = _baseline_design(df)
    X_cpp = _cpp_features(dataset_dir, df)
    rng = np.random.default_rng(42)

    performance_rows: List[Dict[str, float | str]] = []
    fold_rows: List[Dict[str, float | str]] = []
    coefficient_rows: List[Dict[str, float | str]] = []
    ci_rows: List[Dict[str, float | str]] = []
    permutation_rows: List[Dict[str, float | str]] = []
    prediction_store: Dict[Tuple[str, str], np.ndarray] = {}

    for label, window_ms in window_definitions.items():
        X_z = _window_features(latents, times_ms, window_ms)
        X_z_shuffled = X_z.copy()
        rng.shuffle(X_z_shuffled, axis=0)
        z_names = [f"z{k}_mean" for k in range(1, 6)]
        baseline_names = [f"baseline_{i}" for i in range(X_baseline.shape[1])]
        designs: Dict[str, Tuple[np.ndarray, List[str]]] = {
            "baseline": (X_baseline, baseline_names),
            "z-only": (X_z, z_names),
            "baseline+z": (np.concatenate([X_baseline, X_z], axis=1), baseline_names + z_names),
            "baseline+shuffled-z": (np.concatenate([X_baseline, X_z_shuffled], axis=1), baseline_names + [f"shuffled_{name}" for name in z_names]),
        }
        if X_cpp is not None:
            cpp_names = ["cpp_mean", "cpp_late", "cpp_slope"]
            designs["baseline+cpp"] = (np.concatenate([X_baseline, X_cpp], axis=1), baseline_names + cpp_names)
            designs["baseline+cpp+z"] = (np.concatenate([X_baseline, X_cpp, X_z], axis=1), baseline_names + cpp_names + z_names)
        window_fold_rows: List[Dict[str, float | str]] = []
        for model_name, (X, feature_names) in designs.items():
            pred, folds, coefs = _model_cv_predictions(X, log_rt, groups, splits, model_name, label, feature_names)
            prediction_store[(label, model_name)] = pred
            performance_rows.append(
                {
                    "window": label,
                    "model": model_name,
                    "r2": float(r2_score(log_rt, pred)),
                    "corr": float(np.corrcoef(log_rt, pred)[0, 1]),
                }
            )
            window_fold_rows.extend(folds)
            coefficient_rows.extend(coefs)
        by_model_fold = pd.DataFrame(window_fold_rows)
        base = by_model_fold[by_model_fold["model"] == "baseline"][["fold", "r2"]].rename(columns={"r2": "baseline_r2"})
        cpp_base = by_model_fold[by_model_fold["model"] == "baseline+cpp"][["fold", "r2"]].rename(columns={"r2": "cpp_r2"}) if X_cpp is not None else pd.DataFrame(columns=["fold", "cpp_r2"])
        enriched_rows = []
        for row in window_fold_rows:
            fold = row["fold"]
            base_r2 = float(base.loc[base["fold"] == fold, "baseline_r2"].iloc[0])
            cpp_r2 = float(cpp_base.loc[cpp_base["fold"] == fold, "cpp_r2"].iloc[0]) if not cpp_base.empty else np.nan
            row = dict(row)
            row["delta_vs_baseline"] = float(row["r2"] - base_r2)
            row["delta_vs_cpp"] = float(row["r2"] - cpp_r2) if np.isfinite(cpp_r2) else np.nan
            enriched_rows.append(row)
        fold_rows.extend(enriched_rows)

        contrast_defs = [
            ("baseline+z - baseline", "baseline+z", "baseline"),
            ("baseline+shuffled-z - baseline", "baseline+shuffled-z", "baseline"),
        ]
        if X_cpp is not None:
            contrast_defs.append(("baseline+cpp+z - baseline+cpp", "baseline+cpp+z", "baseline+cpp"))
        fold_df_window = pd.DataFrame(enriched_rows)
        for contrast, augmented, reference in contrast_defs:
            paired = []
            for fold in sorted(fold_df_window["fold"].unique()):
                aug = fold_df_window[(fold_df_window["fold"] == fold) & (fold_df_window["model"] == augmented)]["r2"]
                ref = fold_df_window[(fold_df_window["fold"] == fold) & (fold_df_window["model"] == reference)]["r2"]
                if not aug.empty and not ref.empty:
                    paired.append(float(aug.iloc[0] - ref.iloc[0]))
            mean_delta, lo, hi = bootstrap_ci(paired, seed=42)
            ci_rows.append({"window": label, "contrast": contrast, "mean_delta_r2": mean_delta, "ci_lower": lo, "ci_upper": hi, "n_folds": len(paired)})
            obs, p_value = _permutation_test_from_predictions(log_rt, prediction_store[(label, reference)], prediction_store[(label, augmented)], splits, n_perm=n_perm, seed=42)
            permutation_rows.append({"window": label, "contrast": contrast, "observed_delta_r2": obs, "permutation_p_value": p_value, "n_perm": n_perm})

    perf_df = pd.DataFrame(performance_rows)
    fold_df = pd.DataFrame(fold_rows)
    coef_df = pd.DataFrame(coefficient_rows)
    ci_df = pd.DataFrame(ci_rows)
    perm_df = pd.DataFrame(permutation_rows)
    save_dataframe(output_dir / "ridge_rt_performance.csv", perf_df)
    save_dataframe(output_dir / "ridge_rt_fold_metrics.csv", fold_df)
    save_dataframe(output_dir / "ridge_rt_coefficients.csv", coef_df)
    save_dataframe(output_dir / "ridge_delta_r2_ci.csv", ci_df)
    save_dataframe(output_dir / "ridge_permutation_test.csv", perm_df)

    _save_ridge_delta_forestplot(ci_df, output_dir)
    _save_raw_r2_dot_plot(fold_df, output_dir)
    _save_paired_improvement_plot(fold_df, output_dir)
    _save_shuffled_control_plot(fold_df, output_dir)
    _save_z_coefficient_forestplot(coef_df, output_dir)

    summary = {
        "performance": performance_rows,
        "delta_ci": ci_rows,
        "permutation_tests": permutation_rows,
        "outputs": sorted(path.name for path in output_dir.glob("*")),
        "latent_shape": list(latents.shape),
        "rank": 5,
        "cv": "GroupKFold by subject",
    }
    write_json(output_dir / "ridge_rt_summary.json", summary)
    return summary


ridge_results = run_low_rank_r5_ridge_rt_analysis(
    latent_npz=LOW_RANK_R5_LATENT_PATH,
    dataset_dir=DATASET_DIR,
    output_dir=LOW_RANK_R5_REGRESSION_DIR,
)
display(pd.DataFrame(ridge_results["performance"]))
display_saved_figure(LOW_RANK_R5_REGRESSION_DIR / "ridge_delta_r2_forestplot.png", "Delta R² forest plot")
display_saved_figure(LOW_RANK_R5_REGRESSION_DIR / "ridge_raw_r2_dotplot.png", "Raw R² dot plot")
display_saved_figure(LOW_RANK_R5_REGRESSION_DIR / "ridge_paired_improvement_plot.png", "Paired improvement plot")
display_saved_figure(LOW_RANK_R5_REGRESSION_DIR / "ridge_shuffled_control_plot.png", "Shuffled control plot")

## 12 · Results & Next Steps

In [ ]:
def build_results_summary() -> str:
    metrics = json.loads(LOW_RANK_R5_TEST_METRICS.read_text()) if LOW_RANK_R5_TEST_METRICS.exists() else {}
    perf = pd.read_csv(LOW_RANK_R5_REGRESSION_DIR / "ridge_rt_performance.csv")
    ci = pd.read_csv(LOW_RANK_R5_REGRESSION_DIR / "ridge_delta_r2_ci.csv")
    perm = pd.read_csv(LOW_RANK_R5_REGRESSION_DIR / "ridge_permutation_test.csv")
    pivot = perf.pivot(index="window", columns="model", values="r2")
    z_only_best = pivot["z-only"].max() if "z-only" in pivot.columns else np.nan
    baseline_z = ci[ci["contrast"] == "baseline+z - baseline"].copy()
    cpp_z = ci[ci["contrast"] == "baseline+cpp+z - baseline+cpp"].copy()
    shuffled = ci[ci["contrast"] == "baseline+shuffled-z - baseline"].copy()
    best_row = baseline_z.loc[baseline_z["mean_delta_r2"].idxmax()] if not baseline_z.empty else None
    z_improves = bool((baseline_z["mean_delta_r2"] > 0).any()) if not baseline_z.empty else False
    cpp_improves = bool((cpp_z["mean_delta_r2"] > 0).any()) if not cpp_z.empty else False
    shuffled_near = bool((shuffled["ci_lower"].abs() < 0.02).any() or (shuffled["mean_delta_r2"].abs() < 0.02).all()) if not shuffled.empty else False
    perm_sig = perm[(perm["contrast"] == "baseline+z - baseline") & (perm["permutation_p_value"] < 0.05)]
    lines = [
        "### Automatic results interpretation",
        "",
        f"- Rank=5 model training completed and produced a held-out test total loss of `{metrics.get('test_total_loss', float('nan')):.4f}`.",
        f"- Held-out reconstruction loss was `{metrics.get('test_recon_loss', float('nan')):.4f}` and future-prediction loss was `{metrics.get('test_future_loss', float('nan')):.4f}`.",
        f"- The z-only model reached a maximum cross-validated RT-prediction R² of `{z_only_best:.4f}` across the tested response-locked windows.",
        f"- baseline+z exceeded baseline in at least one time window: `{z_improves}`.",
        f"- baseline+cpp+z exceeded baseline+cpp in at least one time window: `{cpp_improves}`.",
        f"- shuffled-z stayed close to the baseline control pattern: `{shuffled_near}`.",
    ]
    if best_row is not None:
        lines.append(f"- The largest baseline+z ΔR² appeared in `{best_row['window']}` with mean ΔR² `{best_row['mean_delta_r2']:.4f}` and 95% CI [`{best_row['ci_lower']:.4f}`, `{best_row['ci_upper']:.4f}`].")
    if len(perm_sig) > 0:
        windows = ", ".join(perm_sig["window"].tolist())
        lines.append(f"- Permutation control suggested non-random baseline+z improvement in: {windows}.")
    else:
        lines.append("- Permutation control did not identify a baseline+z improvement below p < .05 in the current run.")
    lines.extend(
        [
            "",
            "Cautious interpretation: Rank-5 low-rank latent variables captured trial-level neural dynamics that provided incremental RT-predictive information beyond subject/task baselines and conventional CPP summaries. These z variables should not be interpreted as proven CPP components without further converging validation.",
        ]
    )
    return "\n".join(lines)


interpretation_md = build_results_summary()
display(Markdown(interpretation_md))
(RUN_ROOT / "low_rank_r5_interpretation_summary.md").write_text(interpretation_md, encoding="utf-8")

summary = {
    "run_root": str(RUN_ROOT),
    "checkpoint": str(LOW_RANK_R5_BEST_CKPT),
    "test_metrics": str(LOW_RANK_R5_TEST_METRICS),
    "latents": str(LOW_RANK_R5_LATENT_PATH),
    "diagnostics": str(LOW_RANK_R5_DIAGNOSTICS_DIR),
    "regression": str(LOW_RANK_R5_REGRESSION_DIR),
    "rank": 5,
}
write_json(RUN_ROOT / "low_rank_r5_run_summary.json", summary)
print(json.dumps(summary, indent=2))

if LOW_RANK_R5_TEST_METRICS.exists():
    print("\\nHeld-out test metrics:")
    print(LOW_RANK_R5_TEST_METRICS.read_text())

print("\\nNext comparisons: Rank=3 vs Rank=5 vs compact full-rank recurrent baselines, using the same train/validation/test split policy.")

## Cleanup Helper

In [ ]:
print(f"This cleanup cell can delete only this notebook run directory:\\n{RUN_ROOT}")
print("To delete it, set CONFIRM_DELETE = 'DELETE_LOW_RANK_R5_RUN' and re-run this cell.")
CONFIRM_DELETE = ""
if CONFIRM_DELETE == "DELETE_LOW_RANK_R5_RUN":
    shutil.rmtree(RUN_ROOT)
    print(f"Deleted: {RUN_ROOT}")
else:
    print("Kept temporary run directory.")